In [12]:
import pandas as pd
import numpy as np
import glob
from VoteRules3D import VoteResult3D

In [13]:
candidate_num = 5
voter_num = 1000
test = VoteResult3D(voter_num, candidate_num, "1D", "normal") #this is num of voters, candidate, dimension, distribution
print(test.OPTcandidate)

Candidate 4


In [14]:
from numpy import random
import numpy as np
import scipy as sp
import math
import matplotlib.pyplot as plt 
import csv
import pandas as pd
import glob

def gen_file(ballots,num,voter_count): 
    #here num signifies how many candidate columns this should generate, voter count is number of voters
    header = ["ballot_id"]
    rank = 1
    while rank < num + 1: 
        header.append(rank)
        rank += 1
    data = []
    vc = 0
    while vc < voter_count: 
        temp_list = [vc]
        for c in ballots[vc]: 
            temp_list.append(c.id)
        data.append(temp_list)
        vc += 1
    
    with open(r'C:\Users\sagbo\Desktop\summer2026\mainresearch\sim_ballots.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(data) 
    
def remove_randoms(file,m): 
    #here m is the number of candidates 
    new_data = []
    with open(file, mode='r', newline='', encoding='utf-8') as file:
        reader = csv.reader(file)
        header = next(reader) #skipping header
        
        for row in reader:
            temp_list = []
            last_index = m + 1
            if random.randint(0,2) == 0: 
                last_index = (m + 1) - random.randint(0, m-1) #randomize how many candidates will be removed 
            temp_list = row[:last_index] 
            new_data.append(temp_list)
    
    return new_data, header
            
def gen_altered(data,header): 
    with open(r'C:\Users\sagbo\Desktop\summer2026\mainresearch\altered_ballots.csv', 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(header)
        writer.writerows(data)  

In [15]:
gen_file(test.ballots,candidate_num,voter_num) 
data, header = remove_randoms('sim_ballots.csv',candidate_num)
gen_altered(data,header)

In [16]:
df = pd.read_csv('sim_ballots.csv')
df.head()

,ballot_id,1,2,3,4,5
0,0,1,4,2,3,0
1,1,1,4,2,3,0
2,2,1,4,0,2,3
3,3,1,0,4,2,3
4,4,4,2,3,1,0


In [17]:
ballots = [(list(row[1:-1]), 1) for row in df.values.tolist()]
from itertools import combinations
candidates = set()
for ballot, weight in ballots:
    for candidate in ballot:
        candidates.add(candidate)
candidates = list(candidates)
print(len(candidates), candidates)
def simulate_copeland_fast(ballots, candidates):
    cand_index = {c: i for i, c in enumerate(candidates)}
    n = len(candidates)
    matrix = np.zeros((n, n))
    
    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_index]
        
        # Ranked candidates vs each other: order on the ballot determines the win
        for c1, c2 in combinations(ranked, 2):
            matrix[cand_index[c1]][cand_index[c2]] += weight
    
    scores = {c: 0 for c in candidates}
    for c1, c2 in combinations(candidates, 2):
        i, j = cand_index[c1], cand_index[c2]
        if matrix[i][j] > matrix[j][i]:
            scores[c1] += 1
            scores[c2] -= 1
        elif matrix[j][i] > matrix[i][j]:
            scores[c2] += 1
            scores[c1] -= 1
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    print("\nCopeland Scores:")
    for candidate, score in sorted_scores:
        print(f"  {candidate}: {score}")
    winner = sorted_scores[0][0]
    print(f"\nCopeland Winner: {winner}")
    return winner

5 [0, 1, 2, 3, 4]


In [18]:
copeland_winner = simulate_copeland_fast(ballots, candidates)


Copeland Scores:
  4: 2
  0: 1
  2: 0
  3: -1
  1: -2

Copeland Winner: 4


In [19]:
def simulate_actual_plurality_veto(ballots, candidates):
    cand_set = set(candidates)

    scores = {c: 0 for c in candidates}
    expanded_voters = []

    for ballot, weight in ballots:
        ranked = [c for c in ballot if c in cand_set]
        scores[ranked[0]] += weight
        for _ in range(weight):
            expanded_voters.append(ranked)

    standing = set(candidates)
    elimination_order = []

    for ranked_ballot in expanded_voters:
        if len(standing) == 1:
            break

        ranked_set = set(ranked_ballot)

        
        for c in reversed(ranked_ballot):
            if c in standing:
                vetoed_this_pass = {c}
                break

        for vetoed in vetoed_this_pass:
            scores[vetoed] -= 1
            if scores[vetoed] <= 0 and vetoed in standing:
                standing.remove(vetoed)
                elimination_order.append(vetoed)

    if standing:
        winner = list(standing)[0]
    else:
        winner = elimination_order[-1]
    print(f"Elimination order: {elimination_order}")
    print(f"Plurality Veto Winner: {winner}")
    return winner, elimination_order

In [20]:
plurality_veto_winner = simulate_actual_plurality_veto(ballots, candidates)

Elimination order: [2, 1, 3, 4]
Plurality Veto Winner: 0


In [21]:
print("Winner of Plurality Veto Non-Truncated: ", 0, "Distortion of Plurality Veto Non-Truncated: ",test.distortion(test.candidates[0])) 
print("Winner of Copeland Non-Truncated: ", 0, "Distortion of Copeland Non-Truncated: ", test.distortion(test.candidates[0]))
print("Winner of Plurality Veto Equally Last", 3, "Distortion of Plurality Veto Equally Last: ", test.distortion(test.candidates[3]))
print("Winner of Plurality Veto Stats",3, "Distortion of Plurality Veto Stats: ", test.distortion(test.candidates[3]))
print("Winner of Copeland 2", 3, "Distortion of Copeland: ", test.distortion(test.candidates[3]))


Winner of Plurality Veto Non-Truncated:  0 Distortion of Plurality Veto Non-Truncated:  2.4215955991339078
Winner of Copeland Non-Truncated:  0 Distortion of Copeland Non-Truncated:  2.4215955991339078
Winner of Plurality Veto Equally Last 3 Distortion of Plurality Veto Equally Last:  1.1400418087639372
Winner of Plurality Veto Stats 3 Distortion of Plurality Veto Stats:  1.1400418087639372
Winner of Copeland 2 3 Distortion of Copeland:  1.1400418087639372
